In [ ]:
import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers

class ArcFaceLayer(layers.Layer):
    """Bypass layer to allow loading of trained ArcFace weights"""
    def __init__(self, n_classes=1, s=30.0, m=0.50, **kwargs):
        super(ArcFaceLayer, self).__init__(**kwargs)
        self.n_classes = n_classes
        self.s = s
        self.m = m

    def build(self, input_shape):
        self.W = self.add_weight(name='W', shape=(input_shape[0][-1], self.n_classes),
                                 initializer='glorot_uniform', trainable=True)

    def call(self, inputs):
        return inputs[0]

def build_inference_model(input_shape=(112, 112, 3), embedding_size=128):
    """Constructs the MobileNetV2 backbone + Embedding Head"""
    base_model = keras.applications.MobileNetV2(
        input_shape=input_shape, include_top=False, weights='imagenet', pooling='avg'
    )
    x = layers.Dense(embedding_size, use_bias=False, name='embedding')(base_model.output)
    x = layers.BatchNormalization(name='embedding_bn')(x)
    return keras.Model(inputs=base_model.input, outputs=x)

def load_trained_model(weights_path):
    """Initializes model and loads .h5 or .dat weights"""
    model = build_inference_model()
    # by_name=True is critical to skip the training-only ArcFaceLayer weights
    model.load_weights(weights_path, by_name=True)
    return model